# IPA Reasoning Coach — Ontology Setup

**Idempotent** — deletes and recreates `IPACoachOntology` on every run.

Requires `IPACoachLH` to already exist with all 19 tables (run `ipa_coach_setup.ipynb` first).

**7 entity types:**
`PronunciationRule`, `SpeechVariety`, `PhoneticContext`, `Transformation`, `LexicalItem`, `Sentence`, `CommunicativeFactor`

**8 relationship types:**
`appliesIn`, `conflictsWith`, `contextFor`, `implements`, `forVariety`, `reducedBy`, `influencedBy`, `exemplifies`

In [ ]:
%pip install semantic-link --quiet --disable-pip-version-check

In [ ]:
import requests, json, base64, time, random, uuid
import sempy.fabric as fabric
from notebookutils import mssparkutils

workspace_id = fabric.get_workspace_id()
token = mssparkutils.credentials.getToken('pbi')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
print(f'Workspace ID: {workspace_id}\n')

# Find existing IPACoachLH — must already exist
print('Looking for IPACoachLH...')
resp = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses',
    headers=headers
)
lakehouses = resp.json().get('value', [])
lh = next((l for l in lakehouses if l['displayName'] == 'IPACoachLH'), None)

if not lh:
    raise RuntimeError('IPACoachLH not found. Run ipa_coach_setup.ipynb first.')

lakehouse_id = lh['id']
print(f'   Found IPACoachLH (id: {lakehouse_id})')
print('\nInfrastructure ready!')

## Step 1: Build ontology definition

Defines 7 entity types with Lakehouse bindings and 8 relationship types with contextualizations.

**Bridge tables used for relationship contextualizations:**
- `RuleVariety` → appliesIn
- `RuleConflict` → conflictsWith
- `PhoneticContext` → contextFor
- `Transformation` → implements + forVariety
- `LexicalItem` → reducedBy
- `RuleFactorImpact` → influencedBy
- `AnnotatedSegment` → exemplifies

In [ ]:
def det_id(name):
    # Deterministic ID from string - stable across re-runs.
    random.seed(name)
    return random.randint(1_000_000_000_000, 9_999_999_999_999)

def b64(obj):
    return base64.b64encode(json.dumps(obj).encode()).decode()

# ── Entity IDs ────────────────────────────────────────────────────────────────
rule_id    = det_id('PronunciationRule')
variety_id = det_id('SpeechVariety')
ctx_id     = det_id('PhoneticContext')
trans_id   = det_id('Transformation')
lex_id     = det_id('LexicalItem')
sent_id    = det_id('Sentence')
factor_id  = det_id('CommunicativeFactor')

# ── Property IDs (column_name → stable numeric ID) ───────────────────────────
rule_props = {k: det_id(f'PronunciationRule_{k}') for k in [
    'rule_id', 'rule_name', 'category', 'description',
    'informal_desc', 'frequency', 'difficulty_cefr'
]}
variety_props = {k: det_id(f'SpeechVariety_{k}') for k in [
    'variety_id', 'variety_name', 'country', 'region'
]}
ctx_props = {k: det_id(f'PhoneticContext_{k}') for k in [
    'context_id', 'environment_desc', 'syllable_position',
    'stress_condition', 'example_word'
]}
trans_props = {k: det_id(f'Transformation_{k}') for k in [
    'transform_id', 'input_segment', 'formal_ipa', 'casual_ipa', 'change_type'
]}
lex_props = {k: det_id(f'LexicalItem_{k}') for k in [
    'word_id', 'lemma', 'surface_form', 'frequency_rank', 'cefr_level', 'register'
]}
sent_props = {k: det_id(f'Sentence_{k}') for k in [
    'sentence_id', 'raw_text', 'register', 'cefr_level', 'topic'
]}
factor_props = {k: det_id(f'CommunicativeFactor_{k}') for k in [
    'factor_id', 'factor_name', 'category', 'description'
]}

# ── Relationship IDs ──────────────────────────────────────────────────────────
rel_rule_variety  = det_id('Rel_PronunciationRule_appliesIn')
rel_rule_conflict = det_id('Rel_PronunciationRule_conflictsWith')
rel_ctx_rule      = det_id('Rel_PhoneticContext_contextFor')
rel_trans_rule    = det_id('Rel_Transformation_implements')
rel_trans_variety = det_id('Rel_Transformation_forVariety')
rel_lex_rule      = det_id('Rel_LexicalItem_reducedBy')
rel_rule_factor   = det_id('Rel_PronunciationRule_influencedBy')
rel_sent_rule     = det_id('Rel_Sentence_exemplifies')

# ── Helper functions (same pattern as Lamna Healthcare lab) ───────────────────
def prop_entry(k, pid, vt):
    return {'id': str(pid), 'name': k, 'valueType': vt,
            'redefines': None, 'baseTypeNamespaceType': None}

def entity_def(eid, name, id_prop, display_prop, props_dict, value_types):
    return {
        'id': str(eid), 'namespace': 'usertypes', 'namespaceType': 'Custom',
        'name': name, 'visibility': 'Visible',
        'entityIdParts': [str(props_dict[id_prop])],
        'displayNamePropertyId': str(props_dict[display_prop]),
        'baseEntityTypeId': None,
        'properties': [prop_entry(k, props_dict[k], vt) for k, vt in value_types.items()],
        'timeseriesProperties': []
    }

def lh_binding(entity_props, table_name):
    bid = str(uuid.uuid4())
    return {
        'id': bid,
        'dataBindingConfiguration': {
            'dataBindingType': 'NonTimeSeries',
            'propertyBindings': [
                {'sourceColumnName': col, 'targetPropertyId': str(pid)}
                for col, pid in entity_props.items()
            ],
            'sourceTableProperties': {
                'sourceType': 'LakehouseTable',
                'workspaceId': workspace_id,
                'itemId': lakehouse_id,
                'sourceTableName': table_name
            }
        }
    }

def rel_def(rid, name, src_eid, tgt_eid):
    return {
        'id': str(rid), 'namespace': 'usertypes', 'namespaceType': 'Custom',
        'name': name,
        'source': {'entityTypeId': str(src_eid)},
        'target': {'entityTypeId': str(tgt_eid)}
    }

def rel_contextualization(table_name, src_col, src_prop_id, tgt_col, tgt_prop_id):
    cid = str(uuid.uuid4())
    return {
        'id': cid,
        'dataBindingTable': {
            'workspaceId': workspace_id,
            'itemId': lakehouse_id,
            'sourceTableName': table_name,
            'sourceType': 'LakehouseTable'
        },
        'sourceKeyRefBindings': [{'sourceColumnName': src_col, 'targetPropertyId': str(src_prop_id)}],
        'targetKeyRefBindings': [{'sourceColumnName': tgt_col, 'targetPropertyId': str(tgt_prop_id)}]
    }

# ── Entity definitions ────────────────────────────────────────────────────────
rule_entity = entity_def(
    rule_id, 'PronunciationRule', 'rule_id', 'rule_name', rule_props,
    {'rule_id': 'String', 'rule_name': 'String', 'category': 'String',
     'description': 'String', 'informal_desc': 'String',
     'frequency': 'String', 'difficulty_cefr': 'String'})

variety_entity = entity_def(
    variety_id, 'SpeechVariety', 'variety_id', 'variety_name', variety_props,
    {'variety_id': 'String', 'variety_name': 'String',
     'country': 'String', 'region': 'String'})

ctx_entity = entity_def(
    ctx_id, 'PhoneticContext', 'context_id', 'environment_desc', ctx_props,
    {'context_id': 'String', 'environment_desc': 'String',
     'syllable_position': 'String', 'stress_condition': 'String', 'example_word': 'String'})

trans_entity = entity_def(
    trans_id, 'Transformation', 'transform_id', 'input_segment', trans_props,
    {'transform_id': 'String', 'input_segment': 'String',
     'formal_ipa': 'String', 'casual_ipa': 'String', 'change_type': 'String'})

lex_entity = entity_def(
    lex_id, 'LexicalItem', 'word_id', 'lemma', lex_props,
    {'word_id': 'String', 'lemma': 'String', 'surface_form': 'String',
     'frequency_rank': 'BigInt', 'cefr_level': 'String', 'register': 'String'})

sent_entity = entity_def(
    sent_id, 'Sentence', 'sentence_id', 'raw_text', sent_props,
    {'sentence_id': 'String', 'raw_text': 'String',
     'register': 'String', 'cefr_level': 'String', 'topic': 'String'})

factor_entity = entity_def(
    factor_id, 'CommunicativeFactor', 'factor_id', 'factor_name', factor_props,
    {'factor_id': 'String', 'factor_name': 'String',
     'category': 'String', 'description': 'String'})

# ── Data bindings (Lakehouse table per entity) ────────────────────────────────
rule_binding    = lh_binding(rule_props,    'PronunciationRule')
variety_binding = lh_binding(variety_props, 'SpeechVariety')
ctx_binding     = lh_binding(ctx_props,     'PhoneticContext')
trans_binding   = lh_binding(trans_props,   'Transformation')
lex_binding     = lh_binding(lex_props,     'LexicalItem')
sent_binding    = lh_binding(sent_props,    'Sentence')
factor_binding  = lh_binding(factor_props,  'CommunicativeFactor')

# ── Relationship definitions ──────────────────────────────────────────────────
# appliesIn: PronunciationRule -> SpeechVariety (via RuleVariety bridge)
rule_variety_rel  = rel_def(rel_rule_variety,  'appliesIn',     rule_id,    variety_id)
# conflictsWith: PronunciationRule -> PronunciationRule (self-ref via RuleConflict)
rule_conflict_rel = rel_def(rel_rule_conflict, 'conflictsWith', rule_id,    rule_id)
# contextFor: PhoneticContext -> PronunciationRule
ctx_rule_rel      = rel_def(rel_ctx_rule,      'contextFor',    ctx_id,     rule_id)
# implements: Transformation -> PronunciationRule
trans_rule_rel    = rel_def(rel_trans_rule,    'implements',    trans_id,   rule_id)
# forVariety: Transformation -> SpeechVariety
trans_variety_rel = rel_def(rel_trans_variety, 'forVariety',    trans_id,   variety_id)
# reducedBy: LexicalItem -> PronunciationRule
lex_rule_rel      = rel_def(rel_lex_rule,      'reducedBy',     lex_id,     rule_id)
# influencedBy: PronunciationRule -> CommunicativeFactor (via RuleFactorImpact)
rule_factor_rel   = rel_def(rel_rule_factor,   'influencedBy',  rule_id,    factor_id)
# exemplifies: Sentence -> PronunciationRule (via AnnotatedSegment)
sent_rule_rel     = rel_def(rel_sent_rule,     'exemplifies',   sent_id,    rule_id)

# ── Relationship contextualizations (bridge table bindings) ───────────────────
rule_variety_ctx  = rel_contextualization(
    'RuleVariety', 'rule_id', rule_props['rule_id'],
    'variety_id', variety_props['variety_id'])

rule_conflict_ctx = rel_contextualization(
    'RuleConflict', 'rule_a', rule_props['rule_id'],
    'rule_b', rule_props['rule_id'])

ctx_rule_ctx = rel_contextualization(
    'PhoneticContext', 'context_id', ctx_props['context_id'],
    'rule_id', rule_props['rule_id'])

trans_rule_ctx = rel_contextualization(
    'Transformation', 'transform_id', trans_props['transform_id'],
    'rule_id', rule_props['rule_id'])

trans_variety_ctx = rel_contextualization(
    'Transformation', 'transform_id', trans_props['transform_id'],
    'variety_id', variety_props['variety_id'])

lex_rule_ctx = rel_contextualization(
    'LexicalItem', 'word_id', lex_props['word_id'],
    'rule_id', rule_props['rule_id'])

rule_factor_ctx = rel_contextualization(
    'RuleFactorImpact', 'rule_id', rule_props['rule_id'],
    'factor_id', factor_props['factor_id'])

sent_rule_ctx = rel_contextualization(
    'AnnotatedSegment', 'sentence_id', sent_props['sentence_id'],
    'rule_id', rule_props['rule_id'])

print('Entity and relationship definitions ready!')
print(f'   7 entities: PronunciationRule, SpeechVariety, PhoneticContext,')
print(f'               Transformation, LexicalItem, Sentence, CommunicativeFactor')
print(f'   8 relationships: appliesIn, conflictsWith, contextFor, implements,')
print(f'                    forVariety, reducedBy, influencedBy, exemplifies')

## Step 2: POST ontology

Deletes any existing `IPACoachOntology`, then POSTs the complete definition.

> Async creation may take 2–10 minutes. The cell polls until Succeeded or Failed.

In [ ]:
ontology_name = 'IPACoachOntology'

# Refresh token
token = mssparkutils.credentials.getToken('pbi')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Delete existing ontology if present
list_resp = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type=Ontology',
    headers=headers
)
existing = [i for i in list_resp.json().get('value', []) if i['displayName'] == ontology_name]
for item in existing:
    del_resp = requests.delete(
        f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item["id"]}',
        headers=headers
    )
    print(f'Deleted existing {item["displayName"]} -> HTTP {del_resp.status_code}')

# Assemble parts
platform_meta = {'metadata': {'type': 'Ontology', 'displayName': ontology_name}}

parts = [
    {'path': '.platform',       'payload': b64(platform_meta), 'payloadType': 'InlineBase64'},
    {'path': 'definition.json', 'payload': b64({}),            'payloadType': 'InlineBase64'},
    # ── Entities + bindings ──────────────────────────────────────────────────
    {'path': f'EntityTypes/{rule_id}/definition.json',
     'payload': b64(rule_entity),    'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{rule_id}/DataBindings/{rule_binding["id"]}.json',
     'payload': b64(rule_binding),   'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{variety_id}/definition.json',
     'payload': b64(variety_entity), 'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{variety_id}/DataBindings/{variety_binding["id"]}.json',
     'payload': b64(variety_binding),'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{ctx_id}/definition.json',
     'payload': b64(ctx_entity),     'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{ctx_id}/DataBindings/{ctx_binding["id"]}.json',
     'payload': b64(ctx_binding),    'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{trans_id}/definition.json',
     'payload': b64(trans_entity),   'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{trans_id}/DataBindings/{trans_binding["id"]}.json',
     'payload': b64(trans_binding),  'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{lex_id}/definition.json',
     'payload': b64(lex_entity),     'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{lex_id}/DataBindings/{lex_binding["id"]}.json',
     'payload': b64(lex_binding),    'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{sent_id}/definition.json',
     'payload': b64(sent_entity),    'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{sent_id}/DataBindings/{sent_binding["id"]}.json',
     'payload': b64(sent_binding),   'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{factor_id}/definition.json',
     'payload': b64(factor_entity),  'payloadType': 'InlineBase64'},
    {'path': f'EntityTypes/{factor_id}/DataBindings/{factor_binding["id"]}.json',
     'payload': b64(factor_binding), 'payloadType': 'InlineBase64'},
    # ── Relationships + contextualizations ───────────────────────────────────
    {'path': f'RelationshipTypes/{rel_rule_variety}/definition.json',
     'payload': b64(rule_variety_rel), 'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_rule_variety}/Contextualizations/{rule_variety_ctx["id"]}.json',
     'payload': b64(rule_variety_ctx), 'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_rule_conflict}/definition.json',
     'payload': b64(rule_conflict_rel),'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_rule_conflict}/Contextualizations/{rule_conflict_ctx["id"]}.json',
     'payload': b64(rule_conflict_ctx),'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_ctx_rule}/definition.json',
     'payload': b64(ctx_rule_rel),     'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_ctx_rule}/Contextualizations/{ctx_rule_ctx["id"]}.json',
     'payload': b64(ctx_rule_ctx),     'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_trans_rule}/definition.json',
     'payload': b64(trans_rule_rel),   'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_trans_rule}/Contextualizations/{trans_rule_ctx["id"]}.json',
     'payload': b64(trans_rule_ctx),   'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_trans_variety}/definition.json',
     'payload': b64(trans_variety_rel),'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_trans_variety}/Contextualizations/{trans_variety_ctx["id"]}.json',
     'payload': b64(trans_variety_ctx),'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_lex_rule}/definition.json',
     'payload': b64(lex_rule_rel),     'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_lex_rule}/Contextualizations/{lex_rule_ctx["id"]}.json',
     'payload': b64(lex_rule_ctx),     'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_rule_factor}/definition.json',
     'payload': b64(rule_factor_rel),  'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_rule_factor}/Contextualizations/{rule_factor_ctx["id"]}.json',
     'payload': b64(rule_factor_ctx),  'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_sent_rule}/definition.json',
     'payload': b64(sent_rule_rel),    'payloadType': 'InlineBase64'},
    {'path': f'RelationshipTypes/{rel_sent_rule}/Contextualizations/{sent_rule_ctx["id"]}.json',
     'payload': b64(sent_rule_ctx),    'payloadType': 'InlineBase64'},
]

payload = {
    'displayName': ontology_name,
    'type': 'Ontology',
    'definition': {'parts': parts}
}

print(f'POSTing {len(parts)} parts for {ontology_name}...')
resp = requests.post(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items',
    headers=headers,
    json=payload
)
print(f'Status: {resp.status_code}')

if resp.status_code == 201:
    print(f'SUCCESS - ontology id: {resp.json().get("id")}')
elif resp.status_code == 202:
    location = resp.headers.get('Location')
    retry_after = int(resp.headers.get('Retry-After', 5))
    print(f'Async - polling {location}')
    for attempt in range(30):
        time.sleep(retry_after)
        poll = requests.get(location, headers=headers)
        status = poll.json().get('status')
        print(f'  attempt {attempt + 1}: {status}')
        if status == 'Succeeded':
            print('SUCCESS')
            break
        elif status == 'Failed':
            print('FAILED - full response:')
            print(json.dumps(poll.json(), indent=2))
            break
else:
    print('ERROR:', resp.text)

## Step 3: Verify

Lists ontologies in the workspace to confirm creation.

> **Next steps**: After this cell completes, Fabric ingests entity instances from the Lakehouse tables. This can take 2–20 minutes. Open `IPACoachOntology` in your workspace, select an entity type (e.g. `PronunciationRule`), and click **Entity type overview** to see instances.

In [ ]:
token = mssparkutils.credentials.getToken('pbi')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

resp = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type=Ontology',
    headers=headers
)

if resp.status_code == 200:
    items = resp.json().get('value', [])
    print(f'Ontologies in workspace ({len(items)}):')
    for item in items:
        marker = ' <-- created now' if item['displayName'] == 'IPACoachOntology' else ''
        print(f'  {item["displayName"]}  (id: {item["id"]}){marker}')
else:
    print('Could not list ontologies:', resp.status_code, resp.text)